# Assignment 2

In [2]:
from math import log, sqrt, exp
from scipy.stats import norm
from scipy.optimize import brentq

### Helper Functions

In [20]:
def bsm_d1(S, K, T, r, sigma):
    """calculate d1 for Black-Scholes"""
    return (log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt(T))

def bsm_d2(d1, T, sigma):
    """calculate d2 for Black Scholes"""
    return d1 - sigma * sqrt(T)

def bsm_call_price(S, K, T, r, sigma):
    """calculates theoretical Black-Scholes call option price"""
    d1 = bsm_d1(S, K, T, r, sigma)
    d2 = bsm_d2(d1, T, sigma)
    return S * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)

def bsm_put_price(S, K, T, r, sigma):
    """calculates theoretical Black-Scholes put option price"""
    d1 = bsm_d1(S, K, T, r, sigma)
    d2 = bsm_d2(d1, T, sigma)
    return K * exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def bsm_put_delta(d1):
    """ calculates put delta"""
    return norm.cdf(d1) - 1



## Question 1
Suppose a European call option has a price 16.6994 today. The stock price is 100, strike is 90, risk free rate r is 5%, no dividend, and 1 year to expiration. Compute the implied volatility by applying Black Scholes Model. 

Round your answer to the nearest integer. For example, if you get implied volatility as 14.678%, input 15.

In [5]:
def calculate_implied_volatility(market_price, S, K, T, r):
    """solves for implied volatility using Brent's method."""
    # objective function: Black-Scholes price minus market price
    objective = lambda sigma: bsm_call_price(S, K, T, r, sigma) - market_price
    
    try:
        # Search for sigma within realistic bounds [0.01% to 500%]
        return brentq(objective, a=1e-4, b=5.0)
    except ValueError:
        # Returns None if market price violates arbitrage bounds
        return None

# Parameters
S = 100       # Current stock price ($)
K = 90       # Strike price ($)
T = 1    # Time to expiration (years)
r = 0.05        # Risk-free interest rate (5%)
market_price = 16.9994  # Observed call option market price ($)

iv = calculate_implied_volatility(market_price, S, K, T, r)
print(f"Implied Volatility: {iv:.2%}")

Implied Volatility: 21.09%


## Question 2

In the following three questions, you are required to hedge options under the given implied volatility surface. Suppose the implied volatility is:
* sigma(K) = 0.3*exp(-2((K/100) - 1))

Please compute the price of a vanilla European put option - P(S = 100, K = 90, T = 1, r = 0.05).

In [22]:
# evaluate the implied volatility at strike 90
def volatility_surface(K):
    return 0.3 * exp(-2 * ((K/100) - 1))

iv_90 = volatility_surface(90.0)
# calculate put price
put_price = bsm_put_price(100, 90, 1, 0.05, iv_90)
print(f"Vanilla European Put Option Price: {round(put_price, 2)}")

Vanilla European Put Option Price: 7.47


## Question 3
Compute the Black-Scholes Hedging Delta Δ delta for the put in previous question. Round your answer to the two digits

In [21]:
put_delta = bsm_put_delta(
    bsm_d1(100, 90, 1, 0.05, iv_90)
)

print(f"Put Delta: {round(put_delta, 2)}")

Put Delta: -0.27
